In [2]:
import os 
import pandas as pd 
import numpy as np

In [3]:
nose_x = 'nose'
nose_y = 'nose.1'
tail_base_x = 'tail_base'
tail_base_y = 'tail_base.1'

In [ ]:
# Define the path to the folder containing your files
folder_path = '/path/to/csv'
file_path = '/path/to/csv'
# Read the CSV file
velocity_folder = '/path/to/csv'

# List all files in the folder
file_list = os.listdir(folder_path)
print(file_list)

['366.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '368.4_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '362.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '366.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '368.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '369.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '362.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '368.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '367.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '369.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '367.4_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '368.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_0

In [ ]:
import pandas as pd
# Bin for your experimental design. this is an example.
bin_edges = [0, 169, 187, 262, 280, 299, 318, 375]
bin_names = ['Habituation', 'CS1', 'ITI1', 'CS2', 'ITI2', 'CS3', 'Post-CS']

scanning_duration_df = pd.DataFrame(columns=['Number'] + bin_names)
freezing_duration_df = pd.DataFrame(columns=['Number'] + bin_names)
rearing_duration_df = pd.DataFrame(columns=['Number'] + bin_names)

In [6]:
import pandas as pd

def trim(dlc):
    # Removing the likelihood columns (based on your provided column indices)
    columns_to_remove = [3, 6]  # Adjust if necessary
    dlc = dlc.drop(dlc.columns[columns_to_remove], axis=1)
    dlc['Frame'] = dlc.index

    print("Columns after removal:", dlc.columns)
    print("Number of columns after removal:", dlc.shape[1])

    # Ensure the correct columns are selected
    expected_columns = ['Frame', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1']  # Adjust if needed

    # Keep only the expected columns
    dlc = dlc[expected_columns]

    # Remove the first row (if necessary)
    dlc = dlc.iloc[1:]

    # Convert columns to numeric values
    dlc = dlc.apply(pd.to_numeric, errors='coerce')

    # Add a 'Second' column by assuming 'Frame' corresponds to frames per second (8 fps as per your code)
    dlc["Second"] = dlc["Frame"] // 8  # Adjust for frame rate if needed

    # Average coordinates for each second
    result = dlc.groupby("Second").agg({
        'Nose': 'mean', 'Nose.1': 'mean',
        'Tail_base': 'mean', 'Tail_base.1': 'mean',
        }).reset_index()

    return result

In [7]:
def calculate_speed(result):
    # Calculate speed for tailbase
    for i in range(1, len(result)):
        prev_row = result.iloc[i - 1]
        cur_row = result.iloc[i]

        # Calculate speed for Tailbase (same as you already had)
        tailbase_speed = np.sqrt((cur_row['Tail_base'] - prev_row['Tail_base'])**2 +
                                 (cur_row['Tail_base.1'] - prev_row['Tail_base.1'])**2)
        result.at[i, 'Tailbase_Speed'] = tailbase_speed

        # Calculate speed for Nose (new code for nose speed)
        nose_speed = np.sqrt((cur_row['Nose'] - prev_row['Nose'])**2 + (cur_row['Nose.1'] - prev_row['Nose.1'])**2)
        result.at[i, 'Nose_Speed'] = nose_speed

    # Perform similar speed calculations for other body parts if needed

    return result


In [8]:
def scanning_dur(bin_data, bin_end):
    scanning_duration = 0
    is_scanning = False
    scanning_start_time = None

    # Iterate over rows within the bin
    for index, row in bin_data.iterrows():
        nose_speed = row['Nose_Speed']
        tailbase_speed = row['Tailbase_Speed']

        # Check if scanning behavior is detected
        if nose_speed > 10 and tailbase_speed < 10:
            if not is_scanning:  # If scanning just started
                is_scanning = True
                scanning_start_time = row['Second']

        else:
            if is_scanning:  # If scanning just ended
                scanning_duration += (row['Second'] - scanning_start_time)
                is_scanning = False  # Reset scanning flag

    # If scanning behavior continues until the end of the bin
    if is_scanning:
        scanning_duration += (bin_end - scanning_start_time)

    scanning_duration = round(scanning_duration * 1.06667, 1)
    return scanning_duration

In [9]:
def freezing_dur(bin_data, bin_end):
        freezing_duration = 0
        is_freezing = False
        freezing_start_time = 0

        # Iterate over rows within the bin
        for index, row in bin_data.iterrows():
            nose_speed = row['Nose_Speed']
            tailbase_speed = row['Tailbase_Speed']

            # Check if freezing behavior is detected
            if nose_speed < 6 and tailbase_speed < 6:
                if not is_freezing:  # If just started
                    is_freezing = True
                    freezing_start_time = row['Second']

            else:
                if is_freezing:
                    freezing_duration += (row['Second'] - freezing_start_time)
                    is_freezing = False

        # If continues until the end of the bin
        if is_freezing:
            freezing_duration += (bin_end - freezing_start_time)

        freezing_duration = round(freezing_duration * 1.06667, 1)
        return freezing_duration

In [10]:
def rearing_dur(bin_data, bin_start, bin_end):
    rearing_duration = 0
    is_rearing = False
    rearing_start_time = 0

    # Iterate over rows within the bin
    for index, row in bin_data.iterrows():
        nose_x = row['Nose']  # x-coordinate of the nose

        # Check if rearing behavior is detected based on nose position (nose.x)
        if nose_x < 53 or nose_x > 300:  # Assuming rearing behavior is based on extreme nose positions
            if not is_rearing:  # If just started rearing
                is_rearing = True
                rearing_start_time = row['Second']
        else:
            if is_rearing:
                rearing_duration += (row['Second'] - rearing_start_time)
                is_rearing = False

    # If continues until the end of the bin
    if is_rearing:
        rearing_duration += (bin_end - rearing_start_time)

    rearing_duration = round(rearing_duration * 1.06667, 1)
    return rearing_duration


In [11]:
def process_file(file_path):
    global scanning_duration_df, freezing_duration_df, rearing_duration_df
    print("Processing file:", file_path)
    dlc = pd.read_csv(file_path, skiprows=1)
    result = pd.DataFrame(trim(dlc))
    result = calculate_speed(result)

    # Save the result to a new CSV file
    result.to_csv(os.path.join(velocity_folder, os.path.basename(file_path) + "-velocity_data.csv"), index=False)

    bin_scanning_duration = []
    bin_freezing_duration = []
    bin_rearing_duration = []

    # quantify scanning
    for i in range(len(bin_edges) - 1):
        bin_start = bin_edges[i]
        bin_end = bin_edges[i + 1]
        # Filter data for current bin
        bin_data = result[(result['Second'] >= bin_start) & (result['Second'] < bin_end)]

        scanning_duration = scanning_dur(bin_data, bin_end)
        freezing_duration = freezing_dur(bin_data, bin_end)
        rearing_duration = rearing_dur(bin_data, bin_start, bin_end)

        bin_scanning_duration.append(scanning_duration)
        bin_freezing_duration.append(freezing_duration)
        bin_rearing_duration.append(rearing_duration)

    # Create a dictionary with scanning durations
    duration_dict = {'Number': os.path.basename(file_path)}
    duration_dict.update(zip(bin_names, bin_scanning_duration))

    freezing_dict = {'Number': os.path.basename(file_path)}
    freezing_dict.update(zip(bin_names, bin_freezing_duration))

    rearing_dict = {'Number': os.path.basename(file_path)}
    rearing_dict.update(zip(bin_names, bin_rearing_duration))

    # Append to respective DataFrames
    scanning_duration_df = pd.concat([scanning_duration_df, pd.DataFrame([duration_dict])], ignore_index=True)
    freezing_duration_df = pd.concat([freezing_duration_df, pd.DataFrame([freezing_dict])], ignore_index=True)
    rearing_duration_df = pd.concat([rearing_duration_df, pd.DataFrame([rearing_dict])], ignore_index=True)



In [12]:
import os

# List all files in the directory
file_list = os.listdir(file_path)

# Loop through each file in the directory
for filename in file_list:
    # Construct the full file path
    full_file_path = os.path.join(file_path, filename)

    # Check if the item is a file (not a directory) and ends with 'filtered.csv'
    # Check if the item is a file (not a directory) and ends with 'filtered.csv'
    if os.path.isfile(full_file_path) and filename.endswith('.csv') and ("velocity" not in filename):
        process_file(full_file_path)

# Replace empty (NaN) values with 0 in the DataFrames
rearing_duration_df = rearing_duration_df.fillna(0)
scanning_duration_df = scanning_duration_df.fillna(0)
freezing_duration_df = freezing_duration_df.fillna(0)

# Save the DataFrames to CSV files with appropriate filenames
scanning_duration_df.to_csv(os.path.join(file_path, "Hab_scan.csv"), index=False)
freezing_duration_df.to_csv(os.path.join(file_path, "Hab_freezing.csv"), index=False)
rearing_duration_df.to_csv(os.path.join(file_path, "Hab_rearing.csv"), index=False)


Processing file: /Users/elliekogan/Desktop/morevideos/csv/366.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/morevideos/csv/368.4_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/morevideos/csv/362.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/morevideos/csv/366.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_04

/var/folders/qz/pmy_02y95f9bnkfj0nt_jdx40000gn/T/ipykernel_36158/1232128237.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scanning_duration_df = pd.concat([scanning_duration_df, pd.DataFrame([duration_dict])], ignore_index=True)
/var/folders/qz/pmy_02y95f9bnkfj0nt_jdx40000gn/T/ipykernel_36158/1232128237.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  freezing_duration_df = pd.concat([freezing_duration_df, pd.DataFrame([freezing_dict])], ignore_index=True)
/var/folders/qz/pmy_02y95f9b

Processing file: /Users/elliekogan/Desktop/morevideos/csv/369.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/morevideos/csv/362.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/morevideos/csv/368.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/morevideos/csv/367.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_04